<a href="https://colab.research.google.com/github/kunalshrivastavapune25/my-notes/blob/main/agentic_ai/first.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!!pip install litellm
import os
import json
from typing import Dict, List
import re
from litellm import completion
# Important!!!
#
# <---- Set your 'OPENAI_API_KEY' as a secret over there with the "key" icon
#
#
import os
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = api_key




In [3]:
"""
Simple AI Agent - End to End (Single File)

This script demonstrates a basic AI agent loop that:
- Uses an LLM for reasoning
- Chooses tools to execute actions
- Maintains memory
- Iterates until termination
"""

import os
import json
import re
from typing import Dict, List
from litellm import completion

# --------------------------------------------------
# OPTIONAL: Create sample files for demo
# --------------------------------------------------
if not os.path.exists("file1.txt"):
    with open("file1.txt", "w") as f:
        f.write("This is file1 content.")

if not os.path.exists("file2.txt"):
    with open("file2.txt", "w") as f:
        f.write("This is file2 content.")

# --------------------------------------------------
# TOOLS
# --------------------------------------------------
def list_files() -> List[str]:
    return os.listdir(".")

def read_file(file_name: str) -> str:
    try:
        with open(file_name, "r") as f:
            return f.read()
    except FileNotFoundError:
        return f"File '{file_name}' not found."

def terminate(message: str):
    print("\nAGENT TERMINATED")
    print(message)

# --------------------------------------------------
# AGENT RULES (SYSTEM PROMPT)
# --------------------------------------------------
agent_rules = [{
    "role": "system",
    "content": """
You are an AI agent that can perform tasks by using available tools.

Available tools:
- list_files() -> List[str]
- read_file(file_name: str) -> str
- terminate(message: str)

Rules:
- If asked about files, list them before reading
- Every response MUST have exactly one action
- Output format:

```action
{
  "tool_name": "tool_name_here",
  "args": {...}
}
```
"""
}]

# --------------------------------------------------
# HELPERS
# --------------------------------------------------
def extract_markdown_block(text: str, block_name: str) -> str:
    pattern = rf"```{block_name}\n(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if not match:
        raise ValueError("Action block not found")
    return match.group(1).strip()

def parse_action(response: str) -> Dict:
    try:
        action_text = extract_markdown_block(response, "action")
        action_json = json.loads(action_text)
        return action_json
    except Exception as e:
        return {
            "tool_name": "error",
            "args": {"message": str(e)}
        }

def generate_response(messages: List[Dict]) -> str:
    response = completion(
        model="gpt-4o-mini",
        messages=messages
    )
    return response["choices"][0]["message"]["content"]

# --------------------------------------------------
# MEMORY (INITIAL USER TASK)
# --------------------------------------------------
memory = [
    {"role": "user", "content": "What files are in this directory and read file1.txt"}
]

# --------------------------------------------------
# AGENT LOOP
# --------------------------------------------------
max_iterations = 10
iterations = 0

while iterations < max_iterations:
    prompt = agent_rules + memory

    print("\n🤖 Agent thinking...")
    response = generate_response(prompt)
    print("Agent response:\n", response)

    action = parse_action(response)

    if action["tool_name"] == "list_files":
        result = {"result": list_files()}

    elif action["tool_name"] == "read_file":
        result = {"result": read_file(action["args"].get("file_name", ""))}

    elif action["tool_name"] == "terminate":
        terminate(action["args"].get("message", "Done"))
        break

    else:
        result = {"error": action["args"].get("message", "Unknown action")}

    print("Action result:", result)

    memory.extend([
        {"role": "assistant", "content": response},
        {"role": "user", "content": json.dumps(result)}
    ])

    iterations += 1



🤖 Agent thinking...
Agent response:
 ```action
{
  "tool_name": "list_files",
  "args": {}
}
```
Action result: {'result': ['.config', 'file2.txt', 'file1.txt', 'sample_data']}

🤖 Agent thinking...
Agent response:
 ```action
{
  "tool_name": "read_file",
  "args": {"file_name": "file1.txt"}
}
```
Action result: {'result': 'This is file1 content.'}

🤖 Agent thinking...
Agent response:
 ```action
{
  "tool_name": "terminate",
  "args": {"message": "File1 content read successfully."}
}
```

AGENT TERMINATED
File1 content read successfully.


STEP 1: Install Required Libraries (Colab Cell)

In [4]:
!pip install litellm


STEP 2: Imports & Setup (Colab Cell)

In [5]:
import os
import json
from typing import Dict, List
import re
from litellm import completion


STEP 3: (Optional) Create Sample Files to Test the Agent

In [6]:
with open("file1.txt", "w") as f:
    f.write("This is file1 content.")

with open("file2.txt", "w") as f:
    f.write("This is file2 content.")


STEP 4: Define the Tools (Environment Functions)

In [7]:
def list_files() -> List[str]:
    """
    List all files in the current directory.
    """
    return os.listdir(".")


def read_file(file_name: str) -> str:
    """
    Read content of a file.
    """
    try:
        with open(file_name, "r") as f:
            return f.read()
    except FileNotFoundError:
        return f"File '{file_name}' not found."


def terminate(message: str):
    """
    Terminate the agent loop.
    """
    print(f"\nAGENT TERMINATED: {message}")


STEP 5: Define Agent Rules (System Prompt)

In [8]:
agent_rules = [{
    "role": "system",
    "content": """
You are an AI agent that can perform tasks by using available tools.

Available tools:
- list_files() -> List[str]: List all files in the current directory.
- read_file(file_name: str) -> str: Read the content of a file.
- terminate(message: str): End the agent loop and print a summary to the user.

Rules:
- If a user asks about files, list them before reading.
- Every response MUST have exactly ONE action.
- Always respond ONLY in the following format:

```action
{
    "tool_name": "tool_name_here",
    "args": {...}
}
"""
}]

STEP 6: Helper Function – Extract Markdown Action Block

In [9]:
def extract_markdown_block(text: str, block_name: str) -> str:
    """
    Extract content inside ```block_name ... ```
    """
    pattern = rf"```{block_name}\n(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if not match:
        raise ValueError("Action block not found")
    return match.group(1).strip()

In [ ]:
STEP 7: Parse the Agent Action

In [10]:
def parse_action(response: str) -> Dict:
    """
    Convert LLM response into structured action.
    """
    try:
        action_text = extract_markdown_block(response, "action")
        action_json = json.loads(action_text)

        if "tool_name" in action_json and "args" in action_json:
            return action_json
        else:
            return {
                "tool_name": "error",
                "args": {"message": "Invalid action format"}
            }

    except Exception as e:
        return {
            "tool_name": "error",
            "args": {"message": str(e)}
        }


STEP 8: LLM Call Function (Generate Response)

In [11]:
def generate_response(messages: List[Dict]) -> str:
    """
    Call the LLM and return response text.
    """
    response = completion(
        model="gpt-4o-mini",
        messages=messages
    )

    return response["choices"][0]["message"]["content"]


STEP 9: Initialize Memory & User Task

In [17]:
memory = [
    {"role": "user", "content": "What files are in this directory?"},
    {"role": "assistant", "content": "```action\n{\"tool_name\":\"list_files\",\"args\":{}}\n```"},
    {"role": "user", "content": "[\"file1.txt\", \"file2.txt\"]"}
]

STEP 10: The FULL Agent Loop (Core Logic)

In [18]:
max_iterations = 10
iterations = 0

while iterations < max_iterations:

    # 1️⃣ Construct Prompt
    prompt = agent_rules + memory

    # 2️⃣ Generate Response
    print("\n🤖 Agent thinking...")
    response = generate_response(prompt)
    print("Agent Response:\n", response)

    # 3️⃣ Parse Action
    action = parse_action(response)

    # 4️⃣ Execute Action
    if action["tool_name"] == "list_files":
        result = {"result": list_files()}

    elif action["tool_name"] == "read_file":
        file_name = action["args"].get("file_name", "")
        result = {"result": read_file(file_name)}

    elif action["tool_name"] == "terminate":
        terminate(action["args"].get("message", "Done"))
        break

    elif action["tool_name"] == "error":
        result = {"error": action["args"]["message"]}

    else:
        result = {"error": f"Unknown tool {action['tool_name']}"}

    print("Action Result:\n", result)

    # 5️⃣ Update Memory
    memory.extend([
        {"role": "assistant", "content": response},
        {"role": "user", "content": json.dumps(result)}
    ])

    # 6️⃣ Termination Check
    if action["tool_name"] == "terminate":
        break

    iterations += 1



🤖 Agent thinking...
Agent Response:
 ```action
{
    "tool_name": "list_files",
    "args": {}
}
```
Action Result:
 {'result': ['.config', 'file2.txt', 'file1.txt', 'sample_data']}

🤖 Agent thinking...
Agent Response:
 ```action
{
    "tool_name": "read_file",
    "args": {"file_name": "file1.txt"}
}
```
Action Result:
 {'result': 'This is file1 content.'}

🤖 Agent thinking...
Agent Response:
 ```action
{
    "tool_name": "read_file",
    "args": {"file_name": "file2.txt"}
}
```
Action Result:
 {'result': 'This is file2 content.'}

🤖 Agent thinking...
Agent Response:
 ```action
{
    "tool_name": "terminate",
    "args": {"message": "Read contents of file1.txt and file2.txt."}
}
```

AGENT TERMINATED: Read contents of file1.txt and file2.txt.
